# 06. The same drone with MCDPL-style syntax

The `MCDP` builder in `codesign.mcdpl` mirrors the paper's concrete syntax (`mcdp { provides ...; requires ...; ... >= ... }`) directly in Python. It is a thin wrapper over `AlgebraicDP`, `FunctionDP`, and `Loop`, but reads almost identically to the MCDPL source in Fig. 48.

This notebook rebuilds the same drone as notebook **01** with this builder. Output is identical.


## Imports

In [1]:
from codesign import MCDP, solve

## Build the drone

In [2]:
# Same physical constants as notebook 01.
ALPHA = 1.8e6      # Li-ion specific energy, J/kg
G = 9.81           # gravity, m/s^2
C_LIFT = 10.0      # actuator coefficient, W per N^2 of lift

# The `with MCDP(...)` context manager builds an internal port and
# constraint table; the final m.build() compiles it to a Loop(FunctionDP).
with MCDP("drone") as m:
    # provides = outer functionality ports (what the user supplies).
    m.provides("endurance", unit="s")
    m.provides("extra_payload", unit="kg")
    m.provides("extra_power", unit="W")
    m.provides("battery_mass", unit="kg")   # loop axis on the F side

    # requires = outer resource ports (what the system needs).
    m.requires("battery_mass", unit="kg")   # loop axis on the R side
    m.requires("report_mass",  unit="kg")   # outer-visible mirror

    # The constraint reads exactly like the paper:
    # battery_mass >= ((battery+payload)*g)^2 * C_LIFT + extra_power) * endurance / alpha
    def battery_mass_eq(f):
        lift = (f["battery_mass"] + f["extra_payload"]) * G
        actuator_power = C_LIFT * lift * lift
        total_power = actuator_power + f["extra_power"]
        energy = total_power * f["endurance"]
        return energy / ALPHA

    # Same expression bound to both R ports.
    m.constraint("battery_mass", battery_mass_eq)
    m.constraint("report_mass",  battery_mass_eq)
    # Close the loop. The Kleene iteration runs on battery_mass and the
    # outer R only exposes report_mass to downstream consumers.
    m.loop_on("battery_mass")

drone = m.build()
drone

DP(loop(drone, axis=battery_mass): endurance:R+[s]×extra_payload:R+[kg]×extra_power:R+[W] -> A[report_mass:R+[kg]])

## Run the same cases as notebook 01

In [3]:
# Identical mission profiles to notebook 01, for a direct comparison.
cases = [
    ("Short, light",   dict(endurance=60.0,   extra_payload=0.10, extra_power=1.0)),
    ("Medium, modest", dict(endurance=300.0,  extra_payload=0.50, extra_power=5.0)),
    ("Longer mission", dict(endurance=600.0,  extra_payload=0.50, extra_power=5.0)),
    ("Infeasible",     dict(endurance=1800.0, extra_payload=1.00, extra_power=10.0)),
]
for label, f in cases:
    result = solve(drone, f, max_iter=80)
    print(f"{label:<16} iters={result.iterations:>3}  "
          f"feasible={result.feasible}  {result.antichain}")

Short, light     iters=  9  feasible=True  Antichain[(report_mass=0.0003564 kg)]
Medium, modest   iters= 22  feasible=True  Antichain[(report_mass=0.04921 kg)]
Longer mission   iters= 41  feasible=True  Antichain[(report_mass=0.1283 kg)]
Infeasible       iters=  8  feasible=False  Antichain[(report_mass=⊤)]


## Compared with notebook 01

The numbers and iteration counts match exactly. The `MCDP` builder is purely syntactic sugar; it produces the same `Loop(FunctionDP(...))` underneath. The advantage is that it reads more like the MCDPL block in the paper, which is the natural notation for a single self-contained problem with provides/requires declarations.

For *modular* composition with named subsystems, the `System` builder in notebook **07** is more idiomatic.
